# De Levenscyclus van een Zaak

**Thema:** Dossiers en parlementaire zaken.

**Kernvragen:** Wat zijn de verschillende soorten zaken (`zaak.soort`), hoe lang duurt het voordat een zaak wordt afgedaan (`afgedaan = true`), en welke zaken zijn gemarkeerd als groot project?

Dit notebook verkent o.a. de silver-tabellen `zaak`, `zaak_activiteit`, `zaak_gerelateerd_vanuit` en `zaak_vervangen_vanuit`.

In [ ]:
from pathlib import Path

import duckdb
import pandas as pd
import matplotlib.pyplot as plt

def find_database(start: Path | None = None) -> Path:
    """Find the repository DuckDB database by walking up from the current folder."""
    current = (start or Path.cwd()).resolve()
    for folder in (current, *current.parents):
        candidate = folder / "tweedekamer.duckdb"
        if candidate.is_file():
            return candidate
    raise FileNotFoundError("Could not find tweedekamer.duckdb from the current directory")


database_path = find_database()
connection = duckdb.connect(str(database_path), read_only=True)


def query_df(sql: str, parameters=None) -> pd.DataFrame:
    """Execute a SQL query and return the result as a pandas DataFrame."""
    return connection.execute(sql, parameters or []).fetchdf()


database_path

## Onderwerp 1

laat de feitentabel zien <br/>
hoeveel zaken worden er afgedaan en niet afgedaan? <br/>
Wat zijn de top 3 zaken die het meest worden afgewezen? <br/>

In [ ]:
df = query_df("""
SELECT *
FROM silver.zaak""")
df

In [ ]:
df = query_df("""
select count(id)
from silver.zaak
where afgedaan = True
""")
df

In [ ]:
df = query_df("""
select count(id)
from silver.zaak
where afgedaan = False
""")
df

In [ ]:
df = query_df(""" 
select soort, titel
from silver.zaak
where afgedaan = False""")
df

In [ ]:
df = query_df("""
select count(*) as "aantal zaken zonder titel"
from silver.zaak
where titel IS NULL""")
df

In [ ]:
df = query_df("""
select count(*) as "hoeveelheid zaken"
from silver.zaak""")
df

In [ ]:
df = query_df("""
select count(soort), soort
from silver.zaak
where afgedaan = False
group by soort
order by count(soort) desc""")
df

In [ ]:
df = query_df("""
select count(soort), soort
from silver.zaak
group by soort
order by count(soort) desc""")
df

In [ ]:
113329/303097

### Conclusie Onderwerp 1
De enige opmerking die er gemaakt kan worden met de data van hierboven is dat er niet bij alle zaken een titel zit, dit geldt voor ongeveer 37% van de zaken. <br/> Voor de rest is er ook antwoord gegeven op de vraag: "Wat zijn de top 3 zaken die het meest worden afgewezen?", namelijk: <br/> <br/>Brief regering <br/> Brief commissie </br> Motie <br/><br/> dit komt natuurlijk doordat er ook meer van die zaken zijn, want er zijn een stuk minder zaken met de soort "Netwerkverkenning", wat wel logisch is.

## Onderwerp 2

...